In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Nehru Nagar, Delhi - DPCC.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,244.46,341.13,26.68,39.04,42.46,65.98,10.03,1.62,24.98,0.24,1.42,79.90,0.30,224.54,992.88,11.04,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,233.29,328.62,25.64,39.59,41.90,83.57,9.28,1.75,26.87,0.23,1.23,83.51,0.30,295.55,992.10,11.21,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,354.46,483.43,73.56,56.59,89.90,98.81,11.05,2.22,25.36,0.37,2.07,75.60,0.30,308.32,992.05,12.24,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,291.96,398.12,77.34,58.47,93.98,88.71,6.21,1.48,18.56,0.37,2.15,83.56,0.30,180.85,990.35,11.78,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,187.46,288.46,22.33,38.27,38.51,59.36,6.98,1.36,16.88,0.22,0.93,87.26,0.30,129.96,990.23,11.04,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,367.50,506.17,46.84,62.98,63.97,47.91,22.55,2.96,56.38,3.83,23.64,47.97,0.43,234.00,989.88,21.32,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,331.88,478.75,75.06,69.17,82.42,50.02,26.01,3.27,50.99,3.54,20.49,49.48,0.37,221.00,991.31,21.00,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,255.08,394.12,39.78,67.96,62.56,40.37,21.60,2.95,50.67,3.02,16.96,48.55,0.37,253.81,991.21,20.75,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,267.88,404.46,58.51,65.14,73.53,34.94,20.77,3.24,53.87,3.24,21.95,48.53,0.37,284.07,992.12,20.66,0.0,0.0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 20)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
NH3          0
SO2          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 20)
          From Date           To Date   PM2.5    PM10     NO    NO2     NOx  \
0  01-01-2025 00:00  02-01-2025 00:00   61.71  341.13  26.68  39.04  42.460   
1  02-01-2025 00:00  03-01-2025 00:00   61.71  328.62  25.64  39.59  41.900   
2  03-01-2025 00:00  04-01-2025 00:00   61.71  156.96  10.78  56.59  26.055   
3  04-01-2025 00:00  05-01-2025 00:00   61.71  398.12  10.78  58.47  26.055   
4  05-01-2025 00:00  06-01-2025 00:00  187.46  288.46  22.33  38.27  38.510   

     NH3    SO2    CO  Ozone  Benzene  Toluene     RH   WS      WD      BP  \
0  65.98  10.03  1.62  24.98     0.24     1.42  79.90  0.3  224.54  992.88   
1  28.05   9.28  1.75  26.87     0.23     1.23  83.51  0.3  295.55  992.10   
2  28.05  11.05  2.22  25.36     0.37     2.07  75.60  0.3  308.32  992.05   
3  28.05   6.21  1.48  18.56     0.37     2.15  83.56  0.3  180.85  990.35   
4  59.36   6.98  1.36  16.88     0.22     0.93  87.26  0.3  129.96  990.23   

      AT   RF  TOT-RF  
0  27.60 

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,-0.170410,1.696139,0.816435,0.244225,0.562882,3.061098,-0.233015,2.008977,-1.541868,1.208988,-0.275209,1.690636,-1.145966,-0.106775,1.816230,0.290530,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,-0.170410,1.568933,0.741672,0.282725,0.533111,-0.046391,-0.408160,2.296725,-1.464111,1.093744,-0.559343,1.972796,-1.145966,0.985055,1.693178,0.290530,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,-0.170410,-0.176567,-0.326576,1.472731,-0.309275,-0.046391,0.005181,3.337044,-1.526234,2.707169,0.696830,1.354545,-1.145966,1.181403,1.685290,-2.554799,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,-0.170410,2.275634,-0.326576,1.604332,-0.309275,-0.046391,-1.125085,1.699095,-1.805995,2.707169,0.816466,1.976705,-1.145966,-0.778540,1.417101,-2.640011,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,2.671125,1.160572,0.503725,0.190324,0.352884,2.518742,-0.945270,1.433482,-1.875112,0.978499,-1.007976,2.265899,-1.145966,-1.561010,1.398170,0.290530,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,-0.170410,-0.176567,2.265687,1.920034,1.706442,1.580678,2.690732,-0.359408,-0.250032,0.056542,0.016403,-0.805036,-0.960428,0.038680,1.342954,-0.872795,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,-0.170410,3.095508,-0.326576,2.353336,2.687320,1.753544,-0.141940,-0.359408,-0.471783,0.056542,0.016403,-0.687013,-1.046061,-0.161205,1.568549,-0.932072,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,-0.170410,2.234960,1.758161,2.268636,1.631481,0.962949,2.468882,-0.359408,-0.484948,0.056542,0.016403,-0.759703,-1.046061,0.343273,1.552773,-0.978383,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,-0.170410,2.340101,3.104614,2.071235,2.214691,0.518086,2.275055,-0.359408,-0.353296,0.056542,0.016403,-0.761266,-1.046061,0.808542,1.696333,-0.995055,0.0,0.0


In [10]:
df.to_excel('Nehrunagar2025.xlsx', index=False)